In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Reading the file 

In [0]:
df = spark.read.format("csv")\
             .option("header", True)\
             .option("inferSchema", True)\
             .load("/Workspace/Users/edwinvictor73@gmail.com/Data-science/raw/ipl_ball_by_ball.csv")

In [0]:
df.limit(10).display()

### Cleaning the column season before filtering as the value in season column is in two different form

In [0]:
df_season_col_cleaned = df.withColumn("season", \
    when(col("season").rlike(r"^\d{4}/\d{2}$"),year(col("date")))\
        .otherwise(col("season")))\
        .withColumn("season", col("season").cast(IntegerType()))\
        .filter(col("season") >= 2020)
        
                        
    

In [0]:
df_season_col_cleaned.limit(10).display()


### Creating new columns for batsmen runs 

In [0]:
df_season_col_cleaned.limit(5).display()

In [0]:
df_total_runs = df_season_col_cleaned.groupby(col("batter"))\
    .agg(sum("batter_runs").alias("total_runs"),\
     sum(when(col("is_powerplay") == 1, col("batter_runs"))
         .otherwise(0)
         ).alias("Runs_scored_in_powerplay"),\
     sum(when(col("is_middle_overs") == 1, col("batter_runs"))
         .otherwise(0)
        ).alias("Runs_scored_in_middle_overs"),\
     sum(when(col("is_death_overs") == 1, col("batter_runs"))
         .otherwise(0)
         ).alias("Runs_scored_in_death_overs")
     )

In [0]:
df_total_runs.sort("Runs_scored_in_death_overs", ascending=False).limit(20).display()

### Batsmen balls faced 

In [0]:
df_season_col_cleaned_1 = df.withColumn("season", \
                            when(col("season").rlike(r"^\d{4}/\d{2}$"), year(col("date")))\
                                .otherwise(col("season")))\
                                .withColumn("season", col("season").cast(IntegerType()))      

In [0]:
df_season_col_filtered_1 = df_season_col_cleaned_1.filter(col("season") >= 2020)

In [0]:
df_balls_faced = (
    df_season_col_filtered_1
    .groupBy("batter")
    .agg(
        count(
            when(
                (col("wides") == 0) & (col("noballs") == 0),
                col("ball_in_over")
            )
        ).alias("total_balls_faced"),

        count(
            when(
                ((col("wides") == 0) & (col("noballs") == 0)) &
                (col("is_powerplay") == 1),
                col("ball_in_over")
            )
        ).alias("balls_faced_in_powerplay"),

        count(
            when(
                ((col("wides") == 0) & (col("noballs") == 0)) &
                 (col("is_middle_overs") == 1),
                 (col("ball_in_over"))
               )
           ).alias("balls_faced_in_middle_overs"),
        
        count(
            when(
                ((col("wides") == 0 ) & (col("noballs") == 0)) &
                 (col("is_death_overs") == 1) , 
                 (col("ball_in_over"))
                ) 
            ).alias("balls_faced_in_death_overs")
    )
)


In [0]:
df_balls_faced.sort(col("total_balls_faced"), ascending=False).limit(10).display()

In [0]:
df_season_col_filtered_1.limit(25).display()

### Dismissals in different phases of the game

In [0]:
df.limit(5).display()

In [0]:
#Cleaning season column 

df_cleaned_season_2 = df.withColumn("season", 
                                    when(
                                        col("season").rlike(r"^\d{4}/\d{2}$"), year(col("date"))
                                       ).otherwise(col("season"))
                                    ).withColumn(
                                        "season", col("season").cast(IntegerType())
                                    ).filter(col("season") >=2020)
                                    

In [0]:
df_dismissals = df_cleaned_season_2.groupby("batter")\
    .agg(
        count(
            when(
                 col("is_wicket") == 1,
                 col("is_wicket")
               )
           ).alias("total_dismissals"),
        
        count(
            when(
                 ((col("is_wicket") == 1) & (col("is_powerplay") == 1)),
                 col("is_wicket")
               )
            ).alias("total_dismissals_in_powerplay"),
        
        count(
            when(
                 ((col("is_wicket") == 1 ) & (col("is_middle_overs") == 1)),
                 col("is_wicket")
               )
            ).alias("total_dismissals_in_middle_overs"),
        count(
            when(
                ((col("is_wicket") == 1) & (col("is_death_overs") == 1)),
                 col("is_wicket") 
                )
            ).alias("total_dismissals_in_death_overs")
     )
               


In [0]:
df_dismissals.sort("total_dismissals", ascending = False).limit(10).display()

In [0]:
df_total_runs.select("batter").distinct().count()


In [0]:
df_dismissals.select("batter").distinct().count()


In [0]:
df_balls_faced.select("batter").distinct().count()

### Joining the dataframes 

In [0]:
batsman_overall_statistics_dataframe =(
    df_total_runs.join(
                        df_balls_faced, on = "batter", how ="left"
                     )
                  .join(
                      df_dismissals, on = "batter", how="left"
                  )
)